## Meralco Bills Analysis

This analysis will be using the meralco bills dataset that I have compiled from 2022 to 2026. 

*Note:* During the 2022 lockdown period, meter readings for my household were not regularly collected. As a result, some bills were delayed and accumulated into later billing periods. This explains the unusually high bill in May of 2022 and the missing records around this period. The data for 2026 is also limited from January to July. These circumstances were considered during the interpretation of data.

*Cost per kWh:* The cost per kWh was derived from the overall amount due divided by the kWh consumption.

This analysis seeks to answer the questions:
1. Does my cost per kWh appear to be increasing or decreasing?
2. Does my electricity consumption appear to be increasing or decreasing?
3. Am I using less electricity but paying more per kWh?


In [ ]:
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Used the 'thousands' argument as the dataset contains commas
bill = pd.read_csv('data/meralco_bills.csv', thousands=',')
bill = bill.rename(columns={
    'YEAR' : 'year',
    'MONTH' : 'month',
    'KWH CONSUMPTION' : 'kwh_cons',
    'AMOUNT DUE' : 'amt_due',
    'COST PER KWH ' : 'cost_per_kwh'
})
# Combine the year and month columns into a date type column 
bill['date'] = bill['year'].astype(str) + '-' + bill['month']

# Convert datatype into datetime
bill['date'] = pd.to_datetime(bill['date'], format='%Y-%B')

# Check if data type is correct
bill['date'].dtype

dtype('<M8[ns]')

In [3]:
bill.describe()

,year,kwh_cons,amt_due,cost_per_kwh,date
count,60.000000,50.000000,50.000000,50.000000,60
mean,2024.000000,69.500000,743.838200,10.950020,2024-06-16 01:36:00
min,2022.000000,41.000000,374.150000,6.380000,2022-01-01 00:00:00
25%,2023.000000,58.000000,589.410000,9.068000,2023-03-24 06:00:00
50%,2024.000000,67.000000,709.385000,11.805000,2024-06-16 00:00:00
75%,2025.000000,75.000000,855.510000,13.080500,2025-09-08 12:00:00
max,2026.000000,156.000000,1570.510000,14.984000,2026-12-01 00:00:00
std,1.426148,20.953057,228.422618,2.592218,NaN


*Note: the max values in kwh_cons and amt_due are substantially higher than their respective means and so potential outliers will be checked later *

In [4]:
bill.head()

,year,month,kwh_cons,amt_due,cost_per_kwh,date
0,2022,January,NaN,NaN,NaN,2022-01-01
1,2022,February,137.0,1316.76,9.611,2022-02-01
2,2022,March,58.0,374.15,6.451,2022-03-01
3,2022,April,NaN,NaN,NaN,2022-04-01
4,2022,May,156.0,1570.51,10.067,2022-05-01


In [192]:
bill.isna().sum()

year             0
month            0
kwh_cons        10
amt_due         10
cost_per_kwh    10
date             0
dtype: int64

#### **Q1. Does my cost per kWh appear to be increasing or decreasing?**

In [130]:
# MONTHLY CHART
figure1 = px.line(
    data_frame=bill,
    x = 'date',
    y = 'cost_per_kwh',
    #color='month',
    labels = {
        'date' : 'Month and Year',
        'cost_per_kwh' : 'Cost per kWh'
    },
    title = 'Monthly cost per kWh from 2022 to 2026',
    template='seaborn'
)

figure1.show()



In [187]:
# YEARLY CHART
yearly_avg_cost =  bill.groupby('year')['cost_per_kwh'].mean().reset_index()
# TRUNCATE THE VALUES FOR LABELS
#yearly_avg_cost['cost_per_kwh'] = yearly_avg_cost['cost_per_kwh'].map(lambda x: np.trunc(x * 100) / 100)

display(yearly_avg_cost)

figure2 = px.line(
    data_frame = yearly_avg_cost,
    x = 'year',
    y = 'cost_per_kwh',
    labels= {
        'year':'Year',
        'cost_per_kwh':'Cost per kWh'
    },
    markers=True,
    text='cost_per_kwh',
    title = 'Yearly Average Cost per kWh from 2022 to 2026',
    template='seaborn'
    )


# Remove the .5 on X axis
figure2.update_xaxes(type = 'category')

# Adjust text position
figure2.update_traces(textposition='bottom center', texttemplate='%{y:.2f}')

figure2.show()

,year,cost_per_kwh
0,2022,7.472875
1,2023,8.455364
2,2024,11.645583
3,2025,12.936417
4,2026,14.246429


Now I want to check if there any potential outliers that might affect the findings by doing a separate analysis without those values from the averages.

In [188]:
# FOR THE OUTLIERS

# FIND THE OUTLIERS
var = 'cost_per_kwh'
Q1 = bill[var].quantile(0.25)
Q3 = bill[var].quantile(0.75)
IQR = Q3 - Q1

lower_boundary = Q1 - 1.5 * IQR
upper_boundary = Q3 + 1.5 * IQR

print('IQR', IQR)
print('Lower Boundary', lower_boundary)
print('Upper Boundary', upper_boundary)

no_outliers_bill = bill[
    (bill[var] >= lower_boundary) & 
    (bill[var] <= upper_boundary)
].reset_index()

display('Outliers:', bill[
    (bill[var] < lower_boundary) |
    (bill[var] > upper_boundary)
].reset_index())

no_outliers_yearly_avg_cost =  no_outliers_bill.groupby('year')['cost_per_kwh'].mean().reset_index()
# TRUNCATE THE VALUES FOR LABELS
#no_outliers_yearly_avg_cost['cost_per_kwh'] = no_outliers_yearly_avg_cost['cost_per_kwh'].map(lambda x: np.trunc(x * 100) / 100)

display(no_outliers_yearly_avg_cost)

# YEARLY CHART
figure2a = px.line(
    data_frame = no_outliers_yearly_avg_cost,
    x = 'year',
    y = 'cost_per_kwh',
    labels= {
        'year':'Year',
        'cost_per_kwh':'Cost per kWh'
    },
    markers=False,
    text='cost_per_kwh',
    title = 'Yearly Average Cost per kWh from 2022 to 2026 Without the Outliers',
    template='seaborn'
    )


# Remove the .5 on X axis
figure2a.update_xaxes(type = 'category')

# Adjust text position
figure2a.update_traces(textposition='bottom center', texttemplate='%{y:.2f}')

figure2a.show()

IQR 4.012499999999999
Lower Boundary 3.0492500000000007
Upper Boundary 19.099249999999998


'Outliers:'

,index,year,month,kwh_cons,amt_due,cost_per_kwh,date


,year,cost_per_kwh
0,2022,7.472875
1,2023,8.455364
2,2024,11.645583
3,2025,12.936417
4,2026,14.246429


##### A1.
| ![Monthly Cost per kWh](images/meralco_bills_analysis/q1-monthly.png) | ![Yearly Average](images/meralco_bills_analysis/q1-yearly.png)
| :---: | :---:
| Figure 1 | Figure 2

&nbsp;
> Looking closer at the monthly costs per kWh, the data shows some fluctuations but ultimately an increasing trend. The yearly average of the cost per kWh also shows an overall upward trend from ₱7.47/kWh in 2022 to ₱14.24/kWh in 2026. This indicates that the cost per kWh has generally increased over the period covered by the dataset.

#### **Q2. Does my electricity consumption appear to be increasing or decreasing?**


In [169]:
# MONTHLY CHART
figure3 = px.line(
    data_frame=bill,
    x = 'date',
    y = 'kwh_cons',
    labels={
        'date': 'Month and Year',
        'kwh_cons': 'kWh Consumption'
    },
    title = 'Monthly kWh Consumption from 2022 to 2026',
    template='seaborn'
)

figure3.show()

In [189]:
# YEARLY CHART
yearly_avg_kwh = bill.groupby('year')['kwh_cons'].mean().reset_index()
# Truncate the yearly average kWh consumption
#yearly_avg_kwh['kwh_cons'] = yearly_avg_kwh['kwh_cons'].map(lambda x: np.trunc(x * 100) / 100)

display(yearly_avg_kwh)

figure4 = px.line(
    data_frame=yearly_avg_kwh,
    x = 'year',
    y = 'kwh_cons',
    labels = {
        'year' : 'Year',
        'kwh_cons' : 'kWh Consumption'
    },
    markers=True,
    text='kwh_cons',
    title = 'Yearly Average kWh Consumption from 2022 to 2026',
    template='seaborn'
)

figure4.update_xaxes(type = 'category')
figure4.update_traces(textposition = 'bottom center', texttemplate='%{y:.2f}')

figure4.show()

,year,kwh_cons
0,2022,87.250000
1,2023,76.636364
2,2024,66.250000
3,2025,60.750000
4,2026,58.571429


Checking if there are any outliers that might affect the findings

In [190]:
# FOR THE OUTLIERS

# FIND THE OUTLIERS
var2 = 'kwh_cons'
Q1_ = bill[var2].quantile(0.25)
Q3_ = bill[var2].quantile(0.75)
IQR_ = Q3_ - Q1_

lower_boundary2 = Q1_ - 1.5 * IQR_
upper_boundary2 = Q3_ + 1.5 * IQR_

print('IQR', IQR_)
print('Lower Boundary', lower_boundary2)
print('Upper Boundary', upper_boundary2)

no_outliers_bill = bill[
    (bill[var2] >= lower_boundary2) & 
    (bill[var2] <= upper_boundary2)
].reset_index()

display('Outliers:', bill[
    (bill[var2] < lower_boundary2) |
    (bill[var2] > upper_boundary2)
].reset_index())

no_outliers_yearly_avg_cons =  no_outliers_bill.groupby('year')['kwh_cons'].mean().reset_index()
# TRUNCATE THE VALUES FOR LABELS
#no_outliers_yearly_avg_cons['kwh_cons'] = no_outliers_yearly_avg_cons['kwh_cons'].map(lambda x: np.trunc(x * 100) / 100)

no_outliers_yearly_avg_cons['difference'] = ((no_outliers_yearly_avg_cons['kwh_cons'] - yearly_avg_kwh['kwh_cons']) / yearly_avg_kwh['kwh_cons']) * 100

display(no_outliers_yearly_avg_cons)

# YEARLY CHART
figure4a = px.line(
    data_frame = no_outliers_yearly_avg_cons,
    x = 'year',
    y = 'kwh_cons',
    labels= {
        'year':'Year',
        'kwh_cons':'kWh Consumption'
    },
    markers=False,
    text='kwh_cons',
    title = 'Yearly Average kWh Consumption from 2022 to 2026 Without the Outliers',
    template='seaborn'
    )


# Remove the .5 on X axis
figure4a.update_xaxes(type = 'category')

# Adjust text position
figure4a.update_traces(textposition='bottom center', texttemplate='%{y:.2f}')

figure4a.show()

IQR 17.0
Lower Boundary 32.5
Upper Boundary 100.5


'Outliers:'

,index,year,month,kwh_cons,amt_due,cost_per_kwh,date
0,1,2022,February,137.0,1316.76,9.611,2022-02-01
1,4,2022,May,156.0,1570.51,10.067,2022-05-01
2,13,2023,February,116.0,837.77,7.222,2023-02-01


,year,kwh_cons,difference
0,2022,67.500000,-22.636103
1,2023,72.700000,-5.136418
2,2024,66.250000,0.000000
3,2025,60.750000,0.000000
4,2026,58.571429,0.000000


##### A2.
| ![Monthly Cost per kWh](images/meralco_bills_analysis/q2-monthly.png) | ![Yearly Average](images/meralco_bills_analysis/q2-yearly.png)
| :---: | :---:
| Figure 3 | Figure 4

| ![Monthly Cost per kWh](images/meralco_bills_analysis/q2-monthly.png)
| :---: |
| Figure 4a

&nbsp;
>The monthly data shows some fluctuations which appears to be due to seasonal changes. The unadjusted yearly average kWh consumption shows an overall downward trend from 87.25 kWh in 2022 to 58.57 kWh in 2026. However, removing three outliers decreases the yearly average of 2022 by approximately 22.63% (from 87.25 kWh to 67.5 kWh) and 2023 by 5.13% (from 76.64 kWh to 72.7 kWh). The adjusted data now shows a climb from 2022 to 2023 followed by a downward trend. Overall, the yearly averages still show a general downward trend over the period covered by the dataset.

#### **Q3. Am I using less electricity but paying more per kWh?**

In [180]:
figure5 = px.scatter(
    bill,
    x = 'kwh_cons',
    y = 'cost_per_kwh',
    labels = {
        'kwh_cons':'kWh Consumption',
        'cost_per_kwh':'Cost per kWh'
    },
    title = 'kWh Consumption v Cost per kWh',
    color='year',
    template='seaborn'
)

figure5.show()

In [181]:
bill[['kwh_cons', 'cost_per_kwh']].corr()

,kwh_cons,cost_per_kwh
kwh_cons,1.000000,-0.322912
cost_per_kwh,-0.322912,1.000000


In [ ]:
# PERCENTAGE CHANGE FROM 2022 TO 2026
cons_change = (
    (yearly_avg_kwh.loc[yearly_avg_kwh['year'] == 2026, 'kwh_cons'].iloc[0]
    - yearly_avg_kwh.loc[yearly_avg_kwh['year'] == 2022, 'kwh_cons'].iloc[0])
    / yearly_avg_kwh.loc[yearly_avg_kwh['year'] == 2022, 'kwh_cons'].iloc[0]
) * 100

cost_change = (
    (yearly_avg_cost.loc[yearly_avg_cost['year'] == 2026, 'cost_per_kwh'].iloc[0]
    - yearly_avg_cost.loc[yearly_avg_cost['year'] == 2022, 'cost_per_kwh'].iloc[0])
    / yearly_avg_cost.loc[yearly_avg_cost['year'] == 2022, 'cost_per_kwh'].iloc[0]
) * 100

print(f'Consumption change: {cons_change:.2f}%')
print(f'Cost per kWh change: {cost_change:.2f}%')

Consumption change: -32.87%
Cost per kWh change: 90.64%


##### A3.
>The scatter plot shows no clear linear relationship between electricity consumption and cost per kWh. Although the correlation coefficient (-0.323) indicates a weak negative linear association, the points form a more complex pattern rather than a clear downward trend. The apparent negative association may also be influenced by the overall changes in both variables over time as well as the various charges included in the cost per kWh.